# Baseline - модель на основе популярности
Базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.  
Рекомендация самых популярных товаров

In [9]:
import os
import sys
from collections import defaultdict
from tqdm import tqdm

import numpy as np
import pandas as pd

sys.path.append('../src')

from data_loader import (
    download_dataset,
    load_orders,
    load_interactions,
    load_items
)

## Загрузка данных

In [2]:
download_dataset()

Датасет загружен ранее


In [3]:
orders_df = load_orders(train=True)
orders_df.head()

,item_id,user_id,created_timestamp,last_status,last_status_timestamp,created_date
0,15535,3423970,2025-01-11 19:20:02.550,delivered_orders,2025-01-16 10:53:03,2025-01-11
1,15535,1207601,2025-03-25 13:11:48.240,canceled_orders,2025-03-29 08:32:43,2025-03-25
2,119241,2340021,2025-05-03 20:46:18.146,canceled_orders,2025-05-12 17:10:27,2025-05-03
3,119241,2356390,2025-05-01 17:53:00.716,delivered_orders,2025-05-14 14:36:20,2025-05-01
4,119241,1490080,2025-04-22 22:59:11.160,canceled_orders,2025-04-26 06:27:28,2025-04-22


In [4]:
tracker_df = load_interactions(train=True)
tracker_df.head()

,item_id,user_id,timestamp,action_type,action_widget,date
0,678602,4211601,2025-05-05 18:11:15,page_view,cart.cartSplit,2025-05-05
1,678602,4211601,2025-05-05 18:12:57,view_description,pdp.characteristics,2025-05-05
2,678602,4211601,2025-05-05 18:12:52,view_description,pdp.richContent,2025-05-05
3,678602,4211601,2025-05-05 10:39:08,page_view,cart.cartSplit,2025-05-05
4,678602,4211601,2025-05-05 18:12:57,view_description,pdp.characteristics,2025-05-05


In [5]:
orders_ddf_test = load_orders(train=False)
orders_ddf_test.head()

,item_id,user_id,created_timestamp,last_status,last_status_timestamp,created_date
0,7510328,981,2025-07-02 08:27:17.686,delivered_orders,2025-07-06 16:14:12,2025-07-02
1,136809690,4670,2025-07-02 17:17:30.423,canceled_orders,2025-07-06 19:35:07,2025-07-02
2,232878073,12341,2025-07-02 15:17:46.393,canceled_orders,2025-07-11 15:36:40,2025-07-02
3,42322137,12860,2025-07-02 00:40:52.503,delivered_orders,2025-07-04 19:44:48,2025-07-02
4,22480781,15330,2025-07-02 07:25:11.226,delivered_orders,2025-07-03 20:07:33,2025-07-02


In [6]:
test_users = orders_ddf_test['user_id'].unique().compute().sort_values().to_list()
test_users[:5]

[1, 21, 30, 31, 51]

## Построение модели на основе популярности

In [7]:
def build_popularity_model(orders_df, tracker_df):
    print("Строим модель популярности...")
    
    # Ввиду ограничений среза данных от оценки по доставленным товарам пришлось отказаться
    #delivered_orders = orders_df[orders_df['last_status'] == 'delivered_orders']
    #print(f"Доставленных заказов: {len(delivered_orders):,}")
    
    #item_popularity = delivered_orders['item_id'].value_counts().compute().to_dict()
    #print(f"Уникальных купленных товаров: {len(item_popularity):,}")

    print(f"Оформленных заказов: {len(orders_df):,}")
    
    item_popularity = orders_df['item_id'].value_counts().compute().to_dict()
    print(f"Уникальных оформленных товаров: {len(item_popularity):,}")

    page_views = tracker_df[tracker_df['action_type'] == 'page_view']
    item_views = page_views['item_id'].value_counts().compute().to_dict()
    print(f"Уникальных просмотренных товаров: {len(item_views):,}")
    
    combined_popularity = defaultdict(float)
    for item_id, count in item_popularity.items():
        combined_popularity[item_id] += count * 3.0
    
    for item_id, count in item_views.items():
        combined_popularity[item_id] += count * 1.0
    
    popular_items = sorted(combined_popularity.items(), key=lambda x: x[1], reverse=True)
    top_items = [item_id for item_id, score in popular_items[:200]]
    
    print(f"Отобрано топ-{len(top_items)} популярных товаров")
    
    top_10_items = popular_items[:10]
    print("\nТоп-10 самых популярных товаров:")
    for i, (item_id, score) in enumerate(top_10_items, 1):
        print(f"  {i}. Товар {item_id}: {score:.1f} баллов")
    
    return top_items

popular_items = build_popularity_model(orders_df, tracker_df)

Строим модель популярности...
Оформленных заказов: 18,887,122
Уникальных оформленных товаров: 4,673,763
Уникальных просмотренных товаров: 4,892,943
Отобрано топ-200 популярных товаров

Топ-10 самых популярных товаров:
  1. Товар 207631139: 270394.0 баллов
  2. Товар 51974017: 256629.0 баллов
  3. Товар 77696741: 192657.0 баллов
  4. Товар 48831711: 167897.0 баллов
  5. Товар 187052809: 155600.0 баллов
  6. Товар 335421166: 142348.0 баллов
  7. Товар 266679380: 111502.0 баллов
  8. Товар 205247231: 100165.0 баллов
  9. Товар 214748195: 94823.0 баллов
  10. Товар 119058499: 93072.0 баллов


In [10]:
pd.DataFrame(
    {"item_id": popular_items}
).to_parquet("../models/popular_items.parquet", index=False)

In [27]:
def build_user_preferences(orders_df, tracker_df, test_users):

    print("Строим пользовательские предпочтения...")

    # Ввиду ограничений среза данных от оценки по доставленным товарам пришлось отказаться
    # delivered_orders = orders_df[orders_df['last_status'] == 'delivered_orders']
    # user_purchased_items = delivered_orders.groupby('user_id')['item_id'].apply(list).to_dict()
    user_purchased_items = orders_df.groupby('user_id')['item_id'].apply(list, meta=('item_id', 'object')).compute().to_dict()
    print(f"Пользователей с покупками: {len(user_purchased_items):,}")
    
    user_viewed_items = tracker_df[tracker_df['action_type'] == 'page_view'].groupby('user_id')['item_id'].apply(list, meta=('item_id', 'object')).compute().to_dict()
    print(f"Пользователей с просмотрами: {len(user_viewed_items):,}")
    
    user_preferences = {}
    users_with_history = 0
    
    for user_id in tqdm(test_users, desc="Обработка предпочтений"):
        preferences = set()
        
        if user_id in user_purchased_items:
            preferences.update(user_purchased_items[user_id])
        
        if user_id in user_viewed_items:
            preferences.update(user_viewed_items[user_id][:50])
        
        user_preferences[user_id] = list(preferences)
        
        if len(preferences) > 0:
            users_with_history += 1
    
    print(f"Построены предпочтения для {len(user_preferences):,} пользователей")
    print(f"Пользователей с историей: {users_with_history:,} ({users_with_history/len(test_users)*100:.1f}%)")
    
    return user_preferences

user_preferences = build_user_preferences(orders_df, tracker_df, test_users)

Строим пользовательские предпочтения...
Пользователей с покупками: 834,137
Пользователей с просмотрами: 1,010,452


Обработка предпочтений: 100%|██████████| 346759/346759 [00:17<00:00, 19489.46it/s]


Построены предпочтения для 346,759 пользователей
Пользователей с историей: 346,744 (100.0%)


In [ ]:
def generate_recommendations(test_users, popular_items, user_preferences):

    print("Генерируем рекомендации...")
    
    recommendations = {}
    
    for user_id in tqdm(test_users, desc="Создание рекомендаций"):
        user_recs = []
        
        user_items = set(user_preferences.get(user_id, []))
        
        available_items = [item for item in popular_items if item not in user_items]
        
        if len(available_items) >= 100:
            user_recs = available_items[:100]
        else:
            user_recs = available_items + popular_items[:100-len(available_items)]
        
        if len(user_recs) < 100:
            remaining = 100 - len(user_recs)
            random_items = np.random.choice(popular_items, remaining, replace=True)
            user_recs.extend(random_items)
        
        recommendations[user_id] = user_recs[:100]
    
    print(f"Сгенерированы рекомендации для {len(recommendations):,} пользователей")
    
    rec_lengths = [len(recs) for recs in recommendations.values()]
    print(f"Средняя длина рекомендаций: {np.mean(rec_lengths):.1f}")
    print(f"Все рекомендации имеют 100 товаров: {all(l == 100 for l in rec_lengths)}")
    
    return recommendations

recommendations = generate_recommendations(test_users, popular_items, user_preferences)

Генерируем рекомендации...


Создание рекомендаций: 100%|██████████| 346759/346759 [00:03<00:00, 94947.21it/s]

Сгенерированы рекомендации для 346,759 пользователей
Средняя длина рекомендаций: 100.0
Все рекомендации имеют 100 товаров: True


## Оценка качества рекомендаций

In [32]:
def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance)[:k]
    if relevance.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, relevance.size + 2))
    return np.sum(relevance * discounts)

def ndcg_at_k(recommended_items, relevant_items, k=100):
    if not relevant_items:
        return None

    relevance = [1 if item in relevant_items else 0 for item in recommended_items[:k]]
    dcg = dcg_at_k(relevance, k)

    ideal_relevance = sorted(relevance, reverse=True)
    idcg = dcg_at_k(ideal_relevance, k)

    return dcg / idcg if idcg > 0 else 0.0

In [38]:
user2items = (
    orders_ddf_test
    .groupby('user_id')['item_id']
    .apply(list, meta=('item_id', 'object'))
    .compute()
    .to_dict()
)

In [39]:
ndcg_scores = []

for user_id, recs in recommendations.items():
    relevant_items = set(user2items.get(user_id, []))
    
    score = ndcg_at_k(recs, relevant_items, k=100)
    
    if score is not None:
        ndcg_scores.append(score)

mean_ndcg = np.mean(ndcg_scores)

print(f"NDCG@100: {mean_ndcg:.5f}")
print(f"Пользователей учтено: {len(ndcg_scores):,}")

NDCG@100: 0.00440
Пользователей учтено: 346,759
